In [2]:
# ============================================
# Setup
# ============================================
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# ============================================
# Step 1: Load data (2024-01 ~ 2026-05)
# ============================================
DATA_DIR = Path("/home/jovyan/Desktop/IDX intern/california")

target_months = [
    "202401",
    "202402",
    "202403",
    "202404",
    "202405",
    "202406",
    "202407",
    "202408",
    "202409",
    "202410",
    "202411",
    "202412",
    "202501",
    "202502",
    "202503",
    "202504",
    "202505",
    "202506",
    "202507",
    "202508",
    "202509",
    "202510",
    "202511",
    "202512",
    "202601",
    "202602",
    "202603",
    "202604",
    "202605",
]

files = sorted(
    [
        f
        for f in DATA_DIR.glob("CRMLSSold*.csv")
        if any(f.name.startswith(f"CRMLSSold{m}") for m in target_months)
    ]
)

print(f"Found {len(files)} files (expecting 29)")

dfs = []
for f in files:
    tmp = pd.read_csv(f, low_memory=False)
    tmp = tmp[
        (tmp["PropertyType"] == "Residential")
        & (tmp["PropertySubType"] == "SingleFamilyResidence")
    ].copy()
    tmp["source_file"] = f.name
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

print(f"Total shape: {df.shape}")
print(f"Date range: {df['CloseDate'].min()} ~ {df['CloseDate'].max()}")


Found 29 files (expecting 29)
Total shape: (320506, 83)
Date range: 2024-01-01 00:00:00 ~ 2026-05-31 00:00:00


In [3]:
# ============================================
# Step 2: ClosePrice cleaning
# ============================================
df = df[df["ClosePrice"] > 0].copy()
df["log_price"] = np.log(df["ClosePrice"])
print(f"After ClosePrice cleaning: {len(df)} rows")


After ClosePrice cleaning: 320503 rows


In [4]:
# ============================================
# Step 3: Coordinate cleaning
# ============================================
sign_flip_mask = (
    (df["Latitude"] >= 32)
    & (df["Latitude"] <= 42)
    & (df["Longitude"] > 0)
    & (-df["Longitude"] >= -125)
    & (-df["Longitude"] <= -114)
)
df.loc[sign_flip_mask, "Longitude"] = -df.loc[sign_flip_mask, "Longitude"]

still_invalid = df[
    (df["Latitude"] < 31.0)
    | (df["Latitude"] > 45)
    | (df["Longitude"] < -125.0)
    | (df["Longitude"] > -113.5)
]
df = df[~df.index.isin(still_invalid.index)].copy()
print(f"After coordinate cleaning: {len(df)} rows")


After coordinate cleaning: 320448 rows


In [5]:
# ============================================
# Step 4: Missing value handling
# ============================================
df["AssociationFee_missing"] = df["AssociationFee"].isna().astype(int)
df["AssociationFee"] = df["AssociationFee"].fillna(0)

for col in ["HighSchoolDistrict", "MLSAreaMajor"]:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].fillna("Missing")

for col in ["ViewYN", "PoolPrivateYN", "AttachedGarageYN"]:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].astype(object).fillna("Missing")

df["Stories_missing"] = df["Stories"].isna().astype(int)
df["Stories"] = df["Stories"].fillna(df["Stories"].median())

numeric_low_missing = [
    "GarageSpaces",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
    "YearBuilt",
    "LivingArea",
    "Longitude",
    "BathroomsTotalInteger",
    "Latitude",
    "ParkingTotal",
]
for col in numeric_low_missing:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].fillna(df[col].median())

df = df.dropna(subset=["City"])

print(f"After missing value handling: {len(df)} rows")


After missing value handling: 320201 rows


In [7]:
# ============================================
# Step 5 (revised): Train + Validation window experiment
# Only Test is fixed at 2026-05
# ============================================
def three_way_time_split_variable_v2(df, train_start, val_start, date_col="CloseDate"):
    train_df = df[(df[date_col] >= train_start) & (df[date_col] < val_start)].copy()
    val_df = df[(df[date_col] >= val_start) & (df[date_col] < "2026-05-01")].copy()
    test_df = df[(df[date_col] >= "2026-05-01") & (df[date_col] < "2026-06-01")].copy()
    return train_df, val_df, test_df


def run_pipeline_for_split_v2(
    df,
    train_start,
    val_start,
    categorical_cols_high_card,
    simple_cat_cols,
    feature_cols_template,
):
    train_df, val_df, test_df = three_way_time_split_variable_v2(
        df, train_start, val_start
    )

    def smoothed_target_encoding(
        train, others, col, target_col="log_price", smoothing=10
    ):
        global_mean = train[target_col].mean()
        agg = train.groupby(col)[target_col].agg(["mean", "count"])
        smoothed = (agg["mean"] * agg["count"] + global_mean * smoothing) / (
            agg["count"] + smoothing
        )
        train[f"{col}_encoded"] = train[col].map(smoothed)
        for o in others:
            o[f"{col}_encoded"] = o[col].map(smoothed).fillna(global_mean)
        return train, others

    for col in categorical_cols_high_card:
        train_df, (val_df, test_df) = smoothed_target_encoding(
            train_df, [val_df, test_df], col
        )

    for col in simple_cat_cols:
        train_df[col] = train_df[col].astype(str)
        val_df[col] = val_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)

    train_df = pd.get_dummies(train_df, columns=simple_cat_cols, drop_first=True)
    val_df = pd.get_dummies(val_df, columns=simple_cat_cols, drop_first=True)
    test_df = pd.get_dummies(test_df, columns=simple_cat_cols, drop_first=True)

    train_df, val_df = train_df.align(val_df, join="left", axis=1, fill_value=0)
    train_df, test_df = train_df.align(test_df, join="left", axis=1, fill_value=0)

    missing_flags = [c for c in train_df.columns if c.endswith("_missing")]
    yn_missing_flags = [f"{c}_missing" for c in simple_cat_cols]
    missing_flags = [c for c in missing_flags if c not in yn_missing_flags]
    missing_flags = [
        c
        for c in missing_flags
        if c
        not in [
            "Longitude_missing",
            "LotSizeSquareFeet_missing",
            "LotSizeArea_missing",
        ]
    ]
    for zero_var_check in ["HighSchoolDistrict_missing", "MLSAreaMajor_missing"]:
        if zero_var_check in missing_flags and train_df[zero_var_check].nunique() <= 1:
            missing_flags.remove(zero_var_check)

    onehot_cols = [
        c
        for c in train_df.columns
        if any(c.startswith(p) for p in simple_cat_cols) and not c.endswith("_missing")
    ]

    location_encoded = [f"{c}_encoded" for c in categorical_cols_high_card]
    feature_cols = (
        feature_cols_template["location_raw_numeric"]
        + location_encoded
        + feature_cols_template["property"]
        + feature_cols_template["lots_financial"]
        + missing_flags
        + onehot_cols
    )
    feature_cols = list(dict.fromkeys(feature_cols))
    feature_cols = [c for c in feature_cols if c in train_df.columns]

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]
    y_train = train_df["log_price"]
    y_val = val_df["log_price"]
    y_test = test_df["log_price"]

    bool_cols = X_train.select_dtypes(include=["bool"]).columns.tolist()
    missing_flag_cols = [c for c in X_train.columns if c.endswith("_missing")]
    no_scale_cols = list(set(bool_cols + missing_flag_cols))
    scale_cols = [c for c in X_train.columns if c not in no_scale_cols]

    scaler = StandardScaler()
    X_train_s = X_train.copy()
    X_val_s = X_val.copy()
    X_test_s = X_test.copy()
    X_train_s[scale_cols] = scaler.fit_transform(X_train[scale_cols])
    X_val_s[scale_cols] = scaler.transform(X_val[scale_cols])
    X_test_s[scale_cols] = scaler.transform(X_test[scale_cols])

    m = LinearRegression()
    m.fit(X_train_s, y_train)

    r2_val = r2_score(y_val, m.predict(X_val_s))
    r2_test = r2_score(y_test, m.predict(X_test_s))

    return len(train_df), len(val_df), r2_val, r2_test


In [9]:
# ============================================
# Step 6 (revised): Run experiment with variable train_start AND val_start
# ============================================
feature_cols_template = {
    "location_raw_numeric": ["Latitude", "Longitude"],
    "property": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "YearBuilt",
        "Stories",
        "GarageSpaces",
        "ParkingTotal",
    ],
    "lots_financial": [
        "AssociationFee",
        "LotSizeSquareFeet",
        "LotSizeAcres",
        "LotSizeArea",
    ],
}
categorical_cols_high_card = [
    "City",
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "HighSchoolDistrict",
]
simple_cat_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN"]

# Try different combinations of (train_start, val_start)
candidate_combos = [
    ("2024-02-01", "2026-02-01"),  # long train (24mo), 3mo val
    ("2024-02-01", "2025-11-01"),  # long train, 6mo val
    ("2025-02-01", "2026-02-01"),  # 13mo train, 3mo val (original)
    ("2025-01-01", "2025-11-01"),  # 10mo train, 6mo val
    ("2025-05-01", "2026-02-01"),  # 9mo train, 3mo val
    ("2025-05-01", "2025-11-01"),  # 6mo train, 6mo val
    ("2025-08-01", "2026-02-01"),  # 6mo train, 3mo val
    ("2025-11-01", "2026-02-01"),  # 3mo train, 3mo val
]

results = []
for train_start, val_start in candidate_combos:
    n_train, n_val, r2_val, r2_test = run_pipeline_for_split_v2(
        df,
        train_start,
        val_start,
        categorical_cols_high_card,
        simple_cat_cols,
        feature_cols_template,
    )
    train_months = round(
        (pd.Timestamp(val_start) - pd.Timestamp(train_start)).days / 30
    )
    val_months = round((pd.Timestamp("2026-05-01") - pd.Timestamp(val_start)).days / 30)
    results.append(
        {
            "train_start": train_start,
            "val_start": val_start,
            "train_months": train_months,
            "val_months": val_months,
            "train_rows": n_train,
            "val_rows": n_val,
            "Val_R2": r2_val,
            "Test_R2": r2_test,
        }
    )
    print(
        f"Train={train_start}~{val_start} ({train_months}mo), Val={val_start}~2026-05 ({val_months}mo): Val R²={r2_val:.4f}, Test R²={r2_test:.4f}"
    )

split_comparison = pd.DataFrame(results)
print("\n=== Summary (sorted by Val R²) ===")
print(split_comparison.sort_values("Val_R2", ascending=False))


Train=2024-02-01~2026-02-01 (24mo), Val=2026-02-01~2026-05 (3mo): Val R²=0.8548, Test R²=0.8643
Train=2024-02-01~2025-11-01 (21mo), Val=2025-11-01~2026-05 (6mo): Val R²=0.8495, Test R²=0.8637
Train=2025-02-01~2026-02-01 (12mo), Val=2026-02-01~2026-05 (3mo): Val R²=0.8528, Test R²=0.8602
Train=2025-01-01~2025-11-01 (10mo), Val=2025-11-01~2026-05 (6mo): Val R²=0.8469, Test R²=0.8591
Train=2025-05-01~2026-02-01 (9mo), Val=2026-02-01~2026-05 (3mo): Val R²=0.8506, Test R²=0.8582
Train=2025-05-01~2025-11-01 (6mo), Val=2025-11-01~2026-05 (6mo): Val R²=0.8422, Test R²=0.8549
Train=2025-08-01~2026-02-01 (6mo), Val=2026-02-01~2026-05 (3mo): Val R²=0.8463, Test R²=0.8535
Train=2025-11-01~2026-02-01 (3mo), Val=2026-02-01~2026-05 (3mo): Val R²=0.8338, Test R²=0.8386

=== Summary (sorted by Val R²) ===
  train_start   val_start  train_months  val_months  train_rows  val_rows  \
0  2024-02-01  2026-02-01            24           3      268132     31739   
2  2025-02-01  2026-02-01            12       